# Group G

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
# This might be needed in order to get plots to display in the exported HTML for submission.
# In any case, please double check that plots display properly before you submit.
import plotly.io as pio
pio.renderers.default='notebook'
df = pd.read_csv("WeatherEvents_Jan2016-Dec2022.csv")

In [ ]:
fig = px.histogram(df, y=df.Type, title="Conteggio totale per tipo di evento")
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

In [ ]:
fig = px.scatter(df, y="LocationLat", x="LocationLng")
fig.update_layout(hovermode=False)
fig.show()

In [ ]:
#Selezionare eventi atmosferici storici che sono avvenuti nell'arco del tempo del nostro dataset e mostrarli con dei grafici (magari lo spostamento di un huragano (grafico animato))

#Grafico HeatMap che mostra l'andamento degli eventi nei vari mesi

In [ ]:
df['StartTime(UTC)'] = pd.to_datetime(df['StartTime(UTC)'], utc=True)
df['EndTime(UTC)'] = pd.to_datetime(df['EndTime(UTC)'], utc=True)

def to_local(group):
    tz_name = group.name 
    group['Local_Time'] = group['StartTime(UTC)'].dt.tz_convert(tz_name).dt.tz_localize(None)
    return group

df = df.groupby('TimeZone', group_keys=False).apply(to_local)

df['Local_Hour'] = df['Local_Time'].dt.hour

In [ ]:
hourly_counts = df.groupby(['Local_Hour', 'Type']).size().reset_index(name='Event_Count')

# Creiamo il grafico
fig = px.line(
    hourly_counts, 
    x = 'Local_Hour', 
    y = 'Event_Count', 
    color = 'Type',
    title = 'Distribuzione Oraria degli Eventi Meteo (Ora Locale)',
    labels = {'Local_Hour': 'Ora del Giorno (0-23)', 'Event_Count': 'Numero di Eventi'},
    template = 'plotly_white'
)

fig.update_layout(xaxis=dict(tickmode='linear', tick0=0, dtick=1))

fig.show()

In [ ]:
hourly_counts['Percentage'] = hourly_counts.groupby('Type')['Event_Count'].transform(
    lambda x: (x / x.sum()) * 100
)

# Creiamo il grafico
fig = px.line(
    hourly_counts, 
    x = 'Local_Hour', 
    y = 'Percentage', 
    color = 'Type',
    title = 'Percentuale Oraria degli Eventi Meteo (Ora Locale)',
    labels = {'Local_Hour': 'Ora del Giorno (0-23)', 'Event_Count': 'Numero di Eventi'},
    template = 'plotly_white'
)

fig.update_layout(xaxis=dict(tickmode='linear', tick0=0, dtick=1))

fig.show()

In [ ]:
df["Year"] = df["StartTime(UTC)"].dt.year
df["Precipitation(mm)"] = df["Precipitation(in)"] * 25.4

mask = df["Type"] == "Rain"
annual_per_station = df.groupby(["Year", "AirportCode"])["Precipitation(mm)"].sum().reset_index()

final_yearly_data = annual_per_station.groupby("Year")["Precipitation(mm)"].mean().reset_index()


fig = px.bar(
    final_yearly_data,
    x = "Year",
    y = "Precipitation(mm)",
    title="Precipitazioni Medie Annuali per Stazione negli USA",
    labels={
        "Precipitation(mm)": "Media Pioggia Annua [mm]", 
        "Year": "Anno"
    },
    template = "plotly_white"
    )

# non mi convince questo grafico
fig.show()

In [ ]:
snow_year = df[df["Type"] == "Snow"].groupby("Year").size().reset_index(name = "SnowCount")
fog_year = df[df["Type"] == "Fog"].groupby("Year").size().reset_index(name = "FogCount")

fig = make_subplots(
    rows=2, cols=1, 
    shared_xaxes=True, 
    vertical_spacing=0.1,
    subplot_titles=("Distribuzione Annuale Neve", "Distribuzione Annuale Nebbia")
)


fig.add_trace(
    go.Bar(
        x = snow_year["Year"],
        y = snow_year["SnowCount"], 
        name = "Neve",
        marker_color = "skyblue"
    ),
    row=1, col=1
)


fig.add_trace(
    go.Bar(
        x = fog_year["Year"],
        y = fog_year["FogCount"], 
        name = "Nebbia",
        marker_color = "lightgrey"
    ),
    row=2, col=1
)

fig.update_layout(
    title_text = "Analisi Eventi Meteo per Anno",
    template = "plotly_white",
    showlegend = False,
    height = 700
)

fig.update_yaxes(title_text="Conteggio Neve", row=1, col=1)
fig.update_yaxes(title_text="Conteggio Nebbia", row=2, col=1)
fig.update_xaxes(title_text="Anno", row=2, col=1)

fig.show()

In [ ]:
rain = df[df["Type"] == "Rain"]
 
# ── 3. Calcola durata e intensità ─────────────────────────────────────────────
rain["DurationHrs"] = (
    rain["EndTime(UTC)"] - rain["StartTime(UTC)"]
).dt.total_seconds() / 3600
 
# Scarta righe senza dati utili
rain = rain[
    (rain["DurationHrs"] > 0) &
    (rain["Precipitation(mm)"] > 0) &
    rain["LocationLat"].notna() &
    rain["LocationLng"].notna()
].copy()
 
# Intensità = pollici/ora  (più alto → pioggia più intensa)
rain["Intensity"] = rain["Precipitation(mm)"] / rain["DurationHrs"]
 
# Anno dal timestamp di inizio

 
# ── 4. Aggrega per anno × stazione (AirportCode) ──────────────────────────────
agg = (
    rain.groupby(["Year", "AirportCode", "City", "State",
                  "LocationLat", "LocationLng"])
    .agg(
        EventCount    = ("EventId",          "count"),
        AvgIntensity  = ("Intensity",        "mean"),
        AvgDurationHr = ("DurationHrs",      "mean"),
        TotalPrecipIn = ("Precipitation(mm)","sum"),
    )
    .reset_index()
)
 
# Clip outlier per la scala colore (99° percentile)
p99 = agg["AvgIntensity"].quantile(0.99)
agg["IntensityClipped"] = agg["AvgIntensity"].clip(upper=p99)
 
agg["Label"] = (
    agg["City"] + ", " + agg["State"] + "<br>" +
    "Airport: " + agg["AirportCode"] + "<br>" +
    "Events: " + agg["EventCount"].astype(str) + "<br>" +
    "Avg intensity: " + agg["AvgIntensity"].round(4).astype(str) + " in/hr<br>" +
    "Avg duration: " + agg["AvgDurationHr"].round(2).astype(str) + " hr<br>" +
    "Total precip: " + agg["TotalPrecipIn"].round(2).astype(str) + " in"
)
 
agg = agg.sort_values("Year")
years = sorted(agg["Year"].unique())
 
# ── 5. Costruisci figura con animazione ───────────────────────────────────────
print("Costruzione figura...")
 
frames = []
for year in years:
    d = agg[agg["Year"] == year]
    frames.append(go.Frame(
        name=str(year),
        data=[go.Scattergeo(
            lat=d["LocationLat"],
            lon=d["LocationLng"],
            mode="markers",
            marker=dict(
                size=d["EventCount"].clip(upper=d["EventCount"].quantile(0.95))
                       .apply(lambda x: 4 + (x / d["EventCount"].max()) * 14),
                color=d["IntensityClipped"],
                colorscale="Blues",
                cmin=0,
                cmax=p99,
                colorbar=dict(
                    title="Intensità<br>(in/hr)",
                    thickness=15,
                    len=0.6,
                ),
                opacity=0.75,
                line=dict(width=0.3, color="white"),
            ),
            text=d["Label"],
            hovertemplate="%{text}<extra></extra>",
        )],
    ))
 
# Frame iniziale (primo anno)
first = agg[agg["Year"] == years[0]]
 
fig = go.Figure(
    data=[go.Scattergeo(
        lat=first["LocationLat"],
        lon=first["LocationLng"],
        mode="markers",
        marker=dict(
            size=first["EventCount"].clip(upper=first["EventCount"].quantile(0.95))
                   .apply(lambda x: 4 + (x / first["EventCount"].max()) * 14),
            color=first["IntensityClipped"],
            colorscale="Blues",
            cmin=0,
            cmax=p99,
            colorbar=dict(
                title="Intensità<br>(in/hr)",
                thickness=15,
                len=0.6,
            ),
            opacity=0.75,
            line=dict(width=0.3, color="white"),
        ),
        text=first["Label"],
        hovertemplate="%{text}<extra></extra>",
    )],
    frames=frames,
)
 
# Slider + pulsanti play/pause
sliders = [dict(
    active=0,
    steps=[dict(
        method="animate",
        args=[[str(y)], dict(
            mode="immediate",
            frame=dict(duration=800, redraw=True),
            transition=dict(duration=300),
        )],
        label=str(y),
    ) for y in years],
    x=0.05, y=0,
    len=0.9,
    currentvalue=dict(prefix="Anno: ", font=dict(size=16)),
    pad=dict(t=50),
)]
 
fig.update_layout(
    title=dict(
        text="🌧 Intensità pioggia negli USA per stazione meteo",
        font=dict(size=20),
        x=0.5,
    ),
    geo=dict(
        scope="usa",
        projection_type="albers usa",
        showland=True,
        landcolor="#1a1a2e",
        showlakes=True,
        lakecolor="#16213e",
        showcoastlines=True,
        coastlinecolor="#444",
        showsubunits=True,
        subunitcolor="#333",
        bgcolor="#0f0f1a",
    ),
    paper_bgcolor="#0f0f1a",
    plot_bgcolor="#0f0f1a",
    font=dict(color="white"),
    updatemenus=[dict(
        type="buttons",
        showactive=False,
        y=0.02,
        x=0.5,
        xanchor="center",
        buttons=[
            dict(label="▶ Play",  method="animate",
                 args=[None, dict(frame=dict(duration=800, redraw=True),
                                  fromcurrent=True,
                                  transition=dict(duration=300))]),
            dict(label="⏸ Pausa", method="animate",
                 args=[[None], dict(frame=dict(duration=0, redraw=False),
                                    mode="immediate",
                                    transition=dict(duration=0))]),
        ],
    )],
    sliders=sliders,
    margin=dict(l=0, r=0, t=60, b=80),
    height=650,
)
 
# Annotazione legenda dimensione marker
fig.add_annotation(
    x=0.01, y=0.12, xref="paper", yref="paper",
    text="● dim. marker = n° eventi",
    showarrow=False,
    font=dict(size=11, color="#aaa"),
)


In [ ]:
rain = df[df["Type"] == "Rain"]
 
# ── 3. Calcola durata e intensità ─────────────────────────────────────────────
rain["DurationHrs"] = (
    rain["EndTime(UTC)"] - rain["StartTime(UTC)"]
).dt.total_seconds() / 3600
 
rain = rain[
    (rain["DurationHrs"] > 0) &
    (rain["Precipitation(mm)"] > 0) &
    rain["State"].notna()
].copy()
 
# Intensità = pollici/ora
rain["Intensity"] = rain["Precipitation(mm)"] / rain["DurationHrs"]

 
# ── 4. Aggrega per anno × stato ───────────────────────────────────────────────
agg = (
    rain.groupby(["Year", "State"])
    .agg(
        EventCount    = ("EventId",           "count"),
        AvgIntensity  = ("Intensity",         "mean"),
        AvgDurationHr = ("DurationHrs",       "mean"),
        TotalPrecipIn = ("Precipitation(in)", "sum"),
    )
    .reset_index()
)
 
# Clip outlier (99° percentile) per la scala colore
p99 = agg["AvgIntensity"].quantile(0.99)
agg["IntensityClipped"] = agg["AvgIntensity"].clip(upper=p99)
 
agg["HoverText"] = (
    "<b>" + agg["State"] + "</b><br>" +
    "Anno: " + agg["Year"].astype(str) + "<br>" +
    "Intensità media: " + agg["AvgIntensity"].round(4).astype(str) + " in/hr<br>" +
    "Durata media: "    + agg["AvgDurationHr"].round(2).astype(str) + " hr<br>" +
    "Precipit. totali: "+ agg["TotalPrecipIn"].round(2).astype(str) + " in<br>" +
    "N° eventi: "       + agg["EventCount"].astype(str)
)
 
agg = agg.sort_values("Year")
years = sorted(agg["Year"].unique())
 
# ── 5. Costruisci figura con animazione ───────────────────────────────────────
print("Costruzione figura...")
 
def make_choropleth(d):
    return go.Choropleth(
        locations=d["State"],
        locationmode="USA-states",
        z=d["IntensityClipped"],
        zmin=0,
        zmax=p99,
        colorscale="Blues",
        colorbar=dict(
            title=dict(text="Intensità<br>(in/hr)", font=dict(size=13)),
            thickness=16,
            len=0.55,
            x=1.0,
        ),
        marker_line_color="#ffffff",
        marker_line_width=0.5,
        text=d["HoverText"],
        hovertemplate="%{text}<extra></extra>",
    )
 
frames = [
    go.Frame(name=str(year), data=[make_choropleth(agg[agg["Year"] == year])])
    for year in years
]
 
fig = go.Figure(
    data=[make_choropleth(agg[agg["Year"] == years[0]])],
    frames=frames,
)
 
# Slider
sliders = [dict(
    active=0,
    steps=[dict(
        method="animate",
        args=[[str(y)], dict(
            mode="immediate",
            frame=dict(duration=900, redraw=True),
            transition=dict(duration=300),
        )],
        label=str(y),
    ) for y in years],
    x=0.05, y=0,
    len=0.90,
    currentvalue=dict(prefix="Anno: ", font=dict(size=17, color="white")),
    pad=dict(t=55),
    font=dict(color="white"),
)]
 
fig.update_layout(
    title=dict(
        text="🌧 Intensità media pioggia negli USA per stato",
        font=dict(size=21, color="white"),
        x=0.5,
    ),
    geo=dict(
        scope="usa",
        projection_type="albers usa",
        showlakes=True,
        lakecolor="#16213e",
        bgcolor="#0f0f1a",
        showcoastlines=False,
        subunitcolor="#555",
    ),
    paper_bgcolor="#0f0f1a",
    plot_bgcolor="#0f0f1a",
    font=dict(color="white"),
    updatemenus=[dict(
        type="buttons",
        showactive=False,
        y=0.02,
        x=0.5,
        xanchor="center",
        buttons=[
            dict(
                label="▶ Play",
                method="animate",
                args=[None, dict(
                    frame=dict(duration=900, redraw=True),
                    fromcurrent=True,
                    transition=dict(duration=300),
                )],
            ),
            dict(
                label="⏸ Pausa",
                method="animate",
                args=[[None], dict(
                    frame=dict(duration=0, redraw=False),
                    mode="immediate",
                    transition=dict(duration=0),
                )],
            ),
        ],
    )],
    sliders=sliders,
    margin=dict(l=0, r=0, t=60, b=90),
    height=620,
)
